In [ ]:
# %%
import pandas as pd
import numpy as np

# %%
data = pd.read_csv('data/boston.csv')
data.head()

# %%
target_col = data.columns[-1]
target_col

# Izdvajamo y (ciljnu promenljivu) i prebacujemo u numpy matricu
y = data[[target_col]].to_numpy()
y[:5]

# %%
# Izdvajamo X (ulazne atribute)
X_raw = data.drop(columns=[target_col])
X_raw.head()

#  NORMALIZACIJA
X_mean = X_raw.mean()
X_mean

# %%
X_std = X_raw.std().replace(0, 1e-9)
X_std

# %%
X_norm = (X_raw - X_mean) / X_std
X_norm.head()

# Prebacujemo normalizovani DataFrame u numpy matricu za brzo mnozenje
X_norm_np = X_norm.to_numpy()
X_norm_np.shape

# %%
# DODAVANJE SLOBODNOG CLANA (BIAS)
m = X_norm_np.shape[0]  # broj redova (kuca)
jedinice = np.ones((m, 1))
jedinice[:5]

# %%
# Spajamo matricu atributa sa kolonom jedinica na kraju
X = np.hstack([X_norm_np, jedinice])
X[:5]

# %%
X.shape

# %%
#JEDNA ITERACIJA GRADIJENTNOG SPUSTA

m, n = X.shape
# Nasumicno inicijalizujemo parametre (tezine) - n elemenata
w = np.random.randn(1, n) * 0.01
w

# %%
# PREDVIDJANJE (X pomnozeno sa transponovanim w)
pred = X @ w.T
pred[:5]

# %%
# GRESKA (razlika izmedju predvidjanja i stvarne cene)
err = pred - y
err[:5]

# %%
# GRADIJENT (Batch metoda - prosek za sve podatke)
grad = (err.T @ X) / m
grad

# %%
# REGULARIZACIJA (L2)
lam = 0.1
reg = lam * w.copy()
reg

# %%
# Gasimo regularizaciju za w0
reg[0, -1] = 0.0
reg

# %%
# Dodajemo kaznu na gradijent
grad = grad + reg
grad

# %%
# AZURIRANJE TEZINA
alpha = 0.01
w_novo = w - alpha * grad
w_novo # Ovo je nase w nakon jedne iteracije ucenja!

# %%
# 3.6 MSE GRESKA (Ispravljena greska sa float!)
loss = float(np.mean(err**2))
loss


# %%
def normalize(X):
    mean = X.mean()
    std = X.std().replace(0, 1e-9)
    X_norm = (X - mean) / std
    return X_norm, mean, std

def add_bias(X):
    m = X.shape[0]
    return np.hstack([X, np.ones((m, 1))])

def learn(data, target_col, lam=0.0, alpha=0.01, max_iter=10000, tol=0.01, method='batch', auto_lr=False, verbose=True):
    y = data[[target_col]].to_numpy()
    X_raw = data.drop(columns=[target_col])

    X_norm, X_mean, X_std = normalize(X_raw)
    X = add_bias(X_norm.to_numpy())
    m, n = X.shape

    w = np.random.randn(1, n) * 0.01
    prev_w = None
    prev_grad = None

    for it in range(max_iter):
        pred = X @ w.T
        err = pred - y

        if method == 'sgd':
            idx = np.random.randint(m)
            grad = err[idx] * X[idx].reshape(1, -1)
        else:
            grad = (err.T @ X) / m

        reg = lam * w.copy()
        reg[0, -1] = 0.0
        grad = grad + reg

        if auto_lr and method == 'batch' and prev_w is not None:
            delta_w = w - prev_w
            delta_grad = grad - prev_grad
            denom = float(delta_grad @ delta_grad.T)
            if denom > 1e-12:
                alpha = abs(float(delta_w @ delta_grad.T) / denom)

        prev_w = w.copy()
        prev_grad = grad.copy()
        w = w - alpha * grad

        grad_norm = float(np.abs(grad).sum())
        loss = float(np.mean(err**2))

        if verbose and it % 500 == 0:
            print(f"iter={it:5d} | grad_norm={grad_norm:.6f} | MSE={loss:.4f} | alpha={alpha:.6f}")

        if grad_norm < tol:
            if verbose:
                print(f"\nKonvergencija u iteraciji {it}")
            break

    return {'w': w, 'X_mean': X_mean, 'X_std': X_std}

def predict(model, data_new):
    X_norm = (data_new - model['X_mean']) / model['X_std']
    X = add_bias(X_norm.to_numpy())
    return (X @ model['w'].T).flatten()


# Test 1: BATCH metoda
print("--- BATCH GRADIJENTNI SPUST ---")
model_batch = learn(data, target_col, lam=0.1, alpha=0.01, method='batch', auto_lr=False)

# %%
# Test 2: SGD metoda (stohasticki)
print("\n--- SGD GRADIJENTNI SPUST ---")
model_sgd = learn(data, target_col, lam=0.1, alpha=0.001, method='sgd', auto_lr=False)

# %%
# Test 3: BONUS - Automatski Learning Rate
print("\n--- AUTO LR (BONUS) ---")
model_auto = learn(data, target_col, lam=0.1, method='batch', auto_lr=True)

# %%
# Predvidjanje i racunanje korenskog MSE-a (RMSE) za model iz 3. testa
preds = predict(model_auto, data.drop(columns=[target_col]))
y_true = data[target_col].to_numpy()

rmse = np.sqrt(np.mean((preds - y_true) ** 2))
print(f"Konacna RMSE greska: {rmse:.4f}")